In [40]:
import torch

In [22]:
base = torch.cuda.memory_allocated()
batch = torch.randn(32,3,224,224, device="cuda")
delta = torch.cuda.memory_allocated() - base
print(f"{delta / 1024**2:.2f} MiB")

18.38 MiB


In [23]:
a = torch.arange(6.).reshape(2,3)
b = a.t().reshape(6)
c = a.reshape(6)

print(b.data_ptr() == a.data_ptr())
print(c.data_ptr() == a.data_ptr())

False
True


# step 1

In [24]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
print("y      :", y)
print("grad_fn:", y.grad_fn)
print("x.grad :", x.grad)     # 아직 None

y      : tensor(9., grad_fn=<PowBackward0>)
grad_fn: <PowBackward0 object at 0x0000021B3264FF40>
x.grad : None


In [25]:
y.backward()
print("x.grad :", x.grad)

x.grad : tensor(6.)


# step2

In [27]:
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(4.0, requires_grad=True)

a = x * w        # 8
b = a + 1        # 9
c = b ** 2       # 81

print("c        :", c.item())
print("c.grad_fn:", c.grad_fn)
print("b.grad_fn:", b.grad_fn)
print("a.grad_fn:", a.grad_fn)
print("x.grad_fn:", x.grad_fn)   # None — 리프

c        : 81.0
c.grad_fn: <PowBackward0 object at 0x0000021B32C88130>
b.grad_fn: <AddBackward0 object at 0x0000021B32C88310>
a.grad_fn: <MulBackward0 object at 0x0000021B32C88340>
x.grad_fn: None


In [28]:
print("x.is_leaf:", x.is_leaf)
print("a.is_leaf:", a.is_leaf)
c.backward()
print("x.grad:", x.grad)   # dc/dx = 2b·w = 2·9·4 = 72
print("w.grad:", w.grad)   # dc/dw = 2b·x = 2·9·2 = 36
print("a.grad:", a.grad)   # None + 경고

x.is_leaf: True
a.is_leaf: False
x.grad: tensor(72.)
w.grad: tensor(36.)
a.grad: None


C:\Users\SBK\AppData\Local\Temp\ipykernel_7356\4134045213.py:6: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:493.)
  print("a.grad:", a.grad)   # None + 경고


In [29]:
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(4.0, requires_grad=True)
a = x * w
a.retain_grad()          # 이 줄이 backward보다 앞에 있어야 한다
c = (a + 1) ** 2
c.backward()
print("a.grad:", a.grad)  # dc/da = 2b = 18

a.grad: tensor(18.)


# step3

In [30]:
x = torch.tensor(3.0, requires_grad=True)

for i in range(3):
    y = x ** 2
    y.backward()
    print(f"{i}회차 x.grad = {x.grad.item()}")   # 6, 12, 18

0회차 x.grad = 6.0
1회차 x.grad = 12.0
2회차 x.grad = 18.0


In [35]:
x = torch.tensor(3.0, requires_grad=True)

for i in range(3):
    if x.grad is not None:
        x.grad.zero_()
    y = x ** 2
    y.backward()
    print(f"{i}회차 x.grad = {x.grad.item()}")   # 6, 6, 6

0회차 x.grad = 6.0
1회차 x.grad = 6.0
2회차 x.grad = 6.0


In [36]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()
y.backward()        # RuntimeError

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

# step4


In [38]:
def f(t):
    return torch.sin(t) * t ** 2

x = torch.tensor(1.3, requires_grad=True)
y = f(x)
y.backward()
analytic = x.grad.item()
print("autograd :", analytic)

autograd : 2.957324266433716


In [39]:
h = 1e-4
with torch.no_grad():
    x0 = torch.tensor(1.3)
    numeric = ((f(x0 + h) - f(x0 - h)) / (2 * h)).item()

print("수치미분 :", numeric)
print("차이     :", abs(analytic - numeric))

수치미분 : 2.957582473754883
차이     : 0.0002582073211669922


# step 5

In [41]:
import torch

torch.cuda.empty_cache()
base = torch.cuda.memory_allocated()

x = torch.randn(4096, 4096, device="cuda", requires_grad=True)
y = x
for _ in range(5):
    y = torch.sin(y)

used = (torch.cuda.memory_allocated() - base) / 1024**2
print(f"grad 있음 : {used:.2f} MiB")

grad 있음 : 384.00 MiB


In [42]:
del x, y
torch.cuda.empty_cache()
base = torch.cuda.memory_allocated()

with torch.no_grad():
    x = torch.randn(4096, 4096, device="cuda")
    y = x
    for _ in range(5):
        y = torch.sin(y)

used = (torch.cuda.memory_allocated() - base) / 1024**2
print(f"no_grad   : {used:.2f} MiB")

del x, y
torch.cuda.empty_cache()

no_grad   : 128.00 MiB


# step 7

In [43]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2      # [1, 4, 9] — 숫자 3개
y.backward()    # RuntimeError

RuntimeError: grad can be implicitly created only for scalar outputs

In [44]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x ** 2).sum()
y.backward()
print(x.grad)      # [2, 4, 6] = 2x

tensor([2., 4., 6.])


In [45]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2
y.backward(gradient=torch.ones_like(y))
print(x.grad)      # [2, 4, 6] — 셀 14와 같다

tensor([2., 4., 6.])


# step 7

In [46]:
w = torch.tensor(0.0, requires_grad=True)
lr = 0.1

for step in range(20):
    loss = (w - 3) ** 2          # w가 3일 때 최소

    if w.grad is not None:
        w.grad.zero_()
    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad         # 갱신은 학습 대상이 아니다

    if step % 5 == 0:
        print(f"step {step:2d}  w={w.item():.4f}  loss={loss.item():.4f}")

print(f"최종 w = {w.item():.4f}")

step  0  w=0.6000  loss=9.0000
step  5  w=2.2136  loss=0.9664
step 10  w=2.7423  loss=0.1038
step 15  w=2.9156  loss=0.0111
최종 w = 2.9654
